In [0]:
from pyspark.sql.functions import *
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, StructType, StructField, ArrayType, DoubleType, IntegerType
import pandas as pd
from pyarrow import *
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression  
from pyspark.ml.feature import *
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pyarrow.parquet as pq

In [0]:
event_log = spark.read.table("hive_metastore.default.event_log_csv")
event_log = event_log.withColumn("EVDATE", to_timestamp("EVDATE", "dd/MM/yyyy HH:mm:ss"))
display(event_log)

In [0]:
df_event = event_log.filter(
    col("ID_TIPOEVENTO5").isNotNull() &            # not null
    (~col("ID_TIPOEVENTO5").isin("Norte", "Sul  "))  # not in the unwanted set
)

In [0]:
code_STYPE = df_event.groupBy("STYPE").count().orderBy(col("count").desc())

display(code_STYPE)

In [0]:
code_TIPOEVENTO1 = df_event.groupBy("ID_TIPOEVENTO1").count().orderBy(col("count").desc())

display(code_TIPOEVENTO1)

In [0]:
code_TIPOEVENTO2 = df_event.groupBy("ID_TIPOEVENTO2").count().orderBy(col("count").desc())

display(code_TIPOEVENTO2)

In [0]:
code_TIPOEVENTO3 = df_event.groupBy("ID_TIPOEVENTO3").count().orderBy(col("count").desc())

display(code_TIPOEVENTO3)

In [0]:
code_TIPOEVENTO4 = df_event.groupBy("ID_TIPOEVENTO4").count().orderBy(col("count").desc())

display(code_TIPOEVENTO4)

In [0]:
code_TIPOEVENTO5 = df_event.groupBy("ID_TIPOEVENTO5").count().orderBy(col("count").desc())

display(code_TIPOEVENTO5)

In [0]:
# list of the event-ID columns
event_cols = [f"ID_TIPOEVENTO{i}" for i in range(1, 6)]

# frequency of every distinct 5-tuple, sorted by count (highest first)
combo_counts = (
    df_event.groupBy(*event_cols).count().orderBy(desc("count"))
)

display(combo_counts)

In [0]:
unique_combos = df_event.select(*event_cols).distinct()
display(unique_combos)

In [0]:
code_EVDESC = df_event.groupBy("EVDESC").count().orderBy(col("count").desc())

display(code_EVDESC)

In [0]:
alarme_df = df_event.filter(
    lower(col("EVDESC")).like("%alto%")      # contains “falha”, ignoring case
)

display(alarme_df)

In [0]:
falha_df = df_event.filter(
    lower(col("EVDESC")).like("%falha%")      # contains “falha”, ignoring case
)

display(falha_df)

In [0]:
# list of the event-ID columns
event_cols = [f"ID_TIPOEVENTO{i}" for i in range(1, 6)]

# frequency of every distinct 5-tuple, sorted by count (highest first)
combo_counts = (
    falha_df.groupBy(*event_cols).count().orderBy(desc("count"))
)

display(combo_counts)

In [0]:

df_event = falha_df.drop(*["ID_EVDATE","ID_AREAGEOGRAFICA","ID_TIPOINSTALACAO","ID_SIGLA","ID_NIVELTENSAO","ID_OBJETO","ID_OPERATOR","NTIME","NTTAGTIME","STTAGMS","INS_COUNT","LOG_TYPE","STYPE","STTAGATR","STTAGCAUSE","STIMEMS","OPR_ID_LOAD","OPR_ID_LOAD","OPR_TM_LOAD","OPR_TIPO_OPERACAO","OPR_STM_FONTE","STATE_NUMBER","EVDESC_ML", "ID_TIPOEVENTO1", "ID_TIPOEVENTO2", "ID_TIPOEVENTO3", "ID_TIPOEVENTO4", "ID_TIPOEVENTO5","ID_EVENT_LOG", "TAGHL0", "TAGHL1", "TAGOPR", "LOGTYPE", "STATENUMBER", "ID_OBJECTO", "ID_PAINEL", "ID_ATRIBUTO", "POLO"])

In [0]:

df_event = df_event.withColumnRenamed('TAG1', 'ID') \
    .withColumnRenamed('EVDATE', 'DATE')

In [0]:
reordered_columns = ["ID"] + [col for col in df_event.columns if col != "ID"]
df_event = df_event.select(reordered_columns)

In [0]:
df_event = df_event.filter(
    (col("ID").substr(2, 1).isin("P", "S")) &  # Check position 2
    (~col("ID").substr(7, 1).isin("-", "9", "4"))  # Check position 7
)

In [0]:
df_event = df_event.withColumn("ID_prefix", substring(col("ID"), 1, 6))

In [0]:
df_event = df_event.withColumn("DATE", to_timestamp("DATE", "dd/MM/yyyy HH:mm:ss"))

In [0]:
df_event = df_event.withColumn(
    "DATE_ISO8601",
    concat_ws(
        "",
        date_format("DATE", "yyyy-MM-dd'T'HH:mm:ss.SSS"),
        lit("+00:00")
    )
)

In [0]:
display(df_event)

In [0]:
df_event = df_event.drop("DATE_ISO8601")

In [0]:
display(df_event)

In [0]:
df_event.select(
    min("DATE").alias("earliest_timestamp"),
    max("DATE").alias("latest_timestamp")
).display()

In [0]:
date_range = df_event.select(
    min("DATE").alias("start_date"),
    max("DATE").alias("end_date")
).collect()[0]

start_date = date_range["start_date"]
end_date = date_range["end_date"]

In [0]:
df = spark.read.table("hive_metastore.default.model_table_weather")

In [0]:
display(df)

In [0]:
df = df.filter((col("DATE") >= start_date) & (col("DATE") <= end_date))

In [0]:
df = df.withColumn("ID_prefix", substring(col("ID"), 1, 6))

In [0]:
def create_time_bucket(df, time_col):
    return df.withColumn(
        "DATE",
        to_timestamp(
            floor(unix_timestamp(col(time_col)) / 900) * 900
        )
    )

df_buck = create_time_bucket(df, "DATE")
df_event_buck = create_time_bucket(df_event, "DATE")

In [0]:
display(df_buck)
display(df_event_buck)


In [0]:
left  = df_buck.alias("l")
right = df_event_buck.alias("r")

joined = (
    left.join(
        broadcast(right),  # if event set is small
        (col("l.ID_prefix") == col("r.ID_prefix")) &
        (col("l.DATE") == col("r.DATE")),
        how="left"
    )
)

In [0]:
final_df = (
    joined.select(
        "l.*",
        when(col("r.ID_prefix").isNotNull(), 1).otherwise(0).alias("has_falha")
    )
)

display(final_df)

In [0]:
# left  = df.alias("l")        
# right  = df_event.alias("r")            # left side alias

# joined = (
#     left.join(
#         broadcast(right),     # broadcast if the event set is small
#         (col("l.ID_prefix") == col("r.ID_prefix")) &
#         (abs(unix_timestamp("l.DATE") - unix_timestamp("r.DATE")) <= 900),
#         how="left"                  # keep every row from df
#     )
#     .withColumn("diff_sec", abs(unix_timestamp("l.DATE") - unix_timestamp("r.DATE")))
# )

# # -----------------------------------------------------------------------------
# # 3.  Per (ID, DATE) in the left table, keep the single closest match
# w = Window.partitionBy("l.ID_prefix", "l.DATE").orderBy("diff_sec")

# closest = joined.withColumn("rn", row_number().over(w)).filter(col("rn") == 1)

# # -----------------------------------------------------------------------------
# # 4.  Build the 0/1 flag and drop helper columns
# final_df = (
#     closest.select(
#         "l.*",                                      # all original columns
#         when(col("r.ID_prefix").isNotNull(), 1).otherwise(0).alias("has_falha")
#     )
# )

# display(final_df)

In [0]:
# final_df = (
#     df.alias("d")                              # main table
#       .join(df_event.alias("f"),           # or df_event.alias("f") if already filtered
#             on=["ID_prefix", "DATE"], how="left")
#       .select(
#           "d.*",                               # every original column
#           when(col("f.ID_prefix").isNotNull(), 1)     # match → 1
#             .otherwise(0)
#             .alias("has_falha")
#       )
# )

# display(final_df)

In [0]:
prefix_len = 6                   # change if you need 1, 2, 4… chars

df_prefixes = (
    df.filter(col("ID").isNotNull())                 # ignore null IDs
      .select(substring("ID", 1, prefix_len).alias("id_prefix"))
      .distinct()
)

df_event_prefixes = (
    df_event.filter(col("ID").isNotNull())
            .select(substring("ID", 1, prefix_len).alias("id_prefix"))
            .distinct()
)

# ── 2. Intersection = prefixes present in *both* dataframes ───────────────
common_prefixes   = df_prefixes.intersect(df_event_prefixes)
common_prefix_cnt = common_prefixes.count()

# ── 3. Display or log the results ─────────────────────────────────────────
display(common_prefixes)

In [0]:
falha_counts = (
    final_df
        .groupBy("has_falha")
        .count()
        .orderBy(desc("count"))   # 1s first, 0s second
)

display(falha_counts)

In [0]:
display(final_df.filter(col("has_falha") == 1))

In [0]:
final_df.filter(col("has_falha") == 1).select("ID_prefix").distinct().count()

In [0]:
final_df.write.format("delta").mode("overwrite").saveAsTable("lreg_df")

### LSTM Experiment

In [0]:
display(final_df)

In [0]:
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_INDEX", handleInvalid="keep")
    for col in ["ID", "CONCELHO", "AM_PM"]
]

pipeline = Pipeline(stages=indexers)
df_indexed = pipeline.fit(final_df).transform(final_df)

In [0]:
display(df_indexed)

Scaling apenas depois do train/test split para evitar data leakage

In [0]:
# Define window spec to order by DATE within each ID
w = Window.partitionBy("ID").orderBy("DATE")

# Add a row number within each ID to track time steps
df_ordered = df_indexed.withColumn("row_num", row_number().over(w))

In [0]:
display(df_ordered)

In [0]:
features_to_sequence = [
    "INTENSITY", "TENSION", "H_LIM_I", "H_LIM_T",
    "MAVERAGE_2H_I", "MAVERAGE_2H_T", "MAVERAGE_1D_I", "MAVERAGE_1D_T",
    "EVENT_COUNT_I", "EVENT_COUNT_T", "TIME_OVER_LIMIT_I", "TIME_OVER_LIMIT_T",
    "DAY_OF_WEEK", "DAY_OF_MONTH", "DAY_OF_YEAR", "HOUR_OF_DAY",
    "ID_INDEX", "AM_PM_INDEX", "CONCELHO_INDEX"
]

Pensar em avaliar a adição de variaçao da tensao/intensidade

In [0]:
df = df_ordered  # Keep everything on the same object

for feature in features_to_sequence:
    for i in range(96):
        df = df.withColumn(f"{feature}_lag_{i}", lag(feature, i).over(w))

In [0]:
for i in range(1, 9):
    df = df.withColumn(f"FAULT_fwd_{i}", lag("has_falha", -i).over(w))

df = df.withColumn("label", greatest(*[col(f"FAULT_fwd_{i}") for i in range(1, 9)]).cast("int"))


In [0]:
step_arrays = [
    array(*[col(f"{feature}_lag_{i}") for feature in features_to_sequence])
    for i in reversed(range(96))  # old → new
]

df = df.withColumn("features", array(*step_arrays))


In [0]:
required_cols = [
    f"{feature}_lag_{i}" for feature in features_to_sequence for i in range(96)
] + [f"FAULT_fwd_{i}" for i in range(1, 9)]

df_clean = df.dropna(subset=required_cols)

df_sequences = df_clean.select("ID", "DATE", "features", "label")


In [0]:
display(df_sequences)


In [0]:
df_sequences.write.format("delta").mode("append").saveAsTable("df_sequences")

In [0]:
df_sequences = spark.read.table("hive_metastore.default.df_sequences")

In [0]:
df_sequences.selectExpr("size(features) AS timesteps", "size(features[0]) AS features_per_step").distinct().show()


In [0]:
df_sequences.groupBy("label").count().orderBy("label").show()

In [0]:
cutoff_date = "2023-11-30"

df_train = df_sequences.filter(col("DATE") < cutoff_date)
df_test  = df_sequences.filter(col("DATE") >= cutoff_date)

In [0]:
df_test = df_test.repartition(100)

In [0]:
#df_test.write.mode("overwrite").parquet("/lstm_test_sequences")


In [0]:
%fs ls

In [0]:
os.listdir("/dbfs/lstm_test_sequences/")

In [0]:
df_test_loaded = spark.read.parquet("/lstm_test_sequences")

In [0]:
display(df_test_loaded)

In [0]:
df_test_small = df_test_loaded.select("features", "label").persist()

In [0]:
dbutils.fs.rm("/lstm_test_minimal", recurse=True)


In [0]:
df_test_small.repartition(10).write.mode("overwrite").parquet("lstm_test_minimal")


In [0]:
df_test_loaded = spark.read.parquet("/lstm_test_minimal")

In [0]:
path = "/dbfs/lstm_test_minimal"

for file in os.listdir(path):
    full_path = os.path.join(path, file)
    if file.endswith(".parquet") and os.path.getsize(full_path) == 0:
        print(f"Deleting empty file: {full_path}")
        os.remove(full_path)

In [0]:
# Load from DBFS path (notice: we drop /dbfs to give native path to Arrow)
table = parquet.read_table("/lstm_test_minimal")


# Convert Arrow table to NumPy arrays
features_list = table.column("features").to_pylist()
labels = table.column("label").to_numpy()

In [0]:
parquet_path = "/dbfs/lstm_test_sequences"
batches = []

for file in os.listdir(parquet_path):
    if file.endswith(".parquet"):
        table = pq.read_table(os.path.join(parquet_path, file))
        df_part = table.to_pandas()
        X_part = np.array(df_part["features"].tolist())
        y_part = df_part["label"].values
        batches.append((X_part, y_part))

# Concatenate all batches
X_test = np.concatenate([b[0] for b in batches], axis=0)
y_test = np.concatenate([b[1] for b in batches], axis=0)

# Save as .npy for reuse
np.save("/dbfs/tmp/X_test.npy", X_test)
np.save("/dbfs/tmp/y_test.npy", y_test)

In [0]:
X_test_list = []
y_test_list = []

for row in df_test.toLocalIterator():
    X_test_list.append(row["features"])
    y_test_list.append(row["label"])

X_test = np.array(X_test_list)
y_test = np.array(y_test_list)

In [0]:
# Save as a file
df_train.write.mode("overwrite").parquet("hive_lstm_train_sequences")

df_train_pd = pd.read_parquet("/dbfs/tmp/lstm_train_sequences")
X_train = np.array(df_train_pd["features"].tolist())
y_train = df_train_pd["label"].values

In [0]:
X_train = np.array(df_train.select("features").toPandas()["features"].tolist())
y_train = df_train.select("label").toPandas()["label"].values

X_test = np.array(df_test.select("features").toPandas()["features"].tolist())
y_test = df_test.select("label").toPandas()["label"].values

In [0]:
display(df_sequences)